# AutoEDA — Automated Exploratory Data Analysis

**Author:** Farooq Shah | Data Scientist & ML Engineer  
**Organization:** DEVSIL (SMC-PRIVATE) LIMITED  


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
warnings.filterwarnings('ignore')

plt.style.use('dark_background')
plt.rcParams.update({'figure.facecolor':'#09090B','axes.facecolor':'#09090B',
                     'axes.edgecolor':'#1F1F23','font.family':'monospace'})

ORANGE, BLUE, GREEN, RED = '#E97316', '#60A5FA', '#34D399', '#F87171'
os.makedirs('graphs and models', exist_ok=True)
print('ready.')

ready.


## 1. Load Dataset

In [8]:


df = pd.read_csv('learning_log.csv')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'learning_log.csv'

## 2. Overview

In [ ]:
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()

print(f'Rows          : {df.shape[0]:,}')
print(f'Columns       : {df.shape[1]}')
print(f'Numeric       : {len(num_cols)}')
print(f'Categorical   : {len(cat_cols)}')
print(f'Missing total : {df.isnull().sum().sum():,}')
print(f'Duplicates    : {df.duplicated().sum()}')
print(f'Memory        : {df.memory_usage(deep=True).sum()/1024:.1f} KB')
print()
print(df.dtypes)

## 3. Descriptive Statistics

In [ ]:
desc = df[num_cols].describe().T
desc['skewness'] = df[num_cols].skew()
desc['kurtosis'] = df[num_cols].kurt()
desc.style.format('{:.3f}').background_gradient(cmap='RdYlGn')

## 4. Distributions

In [ ]:
n, c = len(num_cols), 3
fig, axes = plt.subplots((n+c-1)//c, c, figsize=(16, 4.2*((n+c-1)//c)))
axes = np.array(axes).flatten()

for i, col in enumerate(num_cols):
    data = df[col].dropna()
    axes[i].hist(data, bins=30, color=ORANGE, alpha=0.65, edgecolor='none')
    axes[i].axvline(data.mean(),   color=BLUE,  lw=1.2, ls='--', label=f'mean={data.mean():.2f}')
    axes[i].axvline(data.median(), color=GREEN, lw=1.2, ls=':',  label=f'median={data.median():.2f}')
    axes[i].set_title(col)
    axes[i].legend(fontsize=7)
    axes[i].grid(axis='y', alpha=0.4)

for j in range(i+1, len(axes)): axes[j].set_visible(False)
plt.tight_layout(pad=1.5)
plt.savefig('graphs and models/distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Correlation Heatmap

In [ ]:
if len(num_cols) >= 2:
    corr = df[num_cols].corr()
    mask = np.triu(np.ones_like(corr, dtype=bool))
    s    = max(8, len(num_cols) * 0.9)
    fig, ax = plt.subplots(figsize=(s, s * 0.82))
    sns.heatmap(corr, mask=mask, cmap=sns.diverging_palette(220,15,as_cmap=True),
                annot=True, fmt='.2f', linewidths=0.4, linecolor='#09090B',
                annot_kws={'size':8}, ax=ax, cbar_kws={'shrink':0.65})
    ax.set_title('Pearson Correlation Matrix')
    plt.tight_layout()
    plt.savefig('graphs and models/correlation_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()

## 6. Missing Values

In [ ]:
miss = df.isnull().sum().reset_index()
miss.columns = ['column', 'missing_count']
miss['missing_pct'] = (miss['missing_count'] / len(df) * 100).round(2)
miss = miss[miss['missing_count'] > 0].sort_values('missing_count', ascending=False)

if miss.empty:
    print('No missing values.')
else:
    fig, ax = plt.subplots(figsize=(12, max(4, len(miss)*0.55)))
    ax.barh(miss['column'], miss['missing_pct'],
            color=[RED if p > 10 else ORANGE for p in miss['missing_pct']],
            edgecolor='none', height=0.55)
    ax.axvline(10, color='#27272A', lw=0.8, ls='--')
    ax.set_xlabel('missing %')
    ax.set_title('Missing Values by Column')
    ax.invert_yaxis()
    ax.grid(axis='x', alpha=0.4)
    plt.tight_layout()
    plt.savefig('graphs and models/missing_values.png', dpi=150, bbox_inches='tight')
    plt.show()
    display(miss.reset_index(drop=True))

## 7. Outlier Detection (IQR)

In [ ]:
summary = []
for col in num_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR    = Q3 - Q1
    lo, hi = Q1 - 1.5*IQR, Q3 + 1.5*IQR
    n_out  = ((df[col] < lo) | (df[col] > hi)).sum()
    summary.append({'column':col, 'outliers':n_out,
                    'outlier_pct':round(n_out/len(df)*100,2),
                    'lower_bound':round(lo,3), 'upper_bound':round(hi,3)})

display(pd.DataFrame(summary).sort_values('outliers',ascending=False).reset_index(drop=True))

n, c = len(num_cols), 3
fig, axes = plt.subplots((n+c-1)//c, c, figsize=(16, 4*((n+c-1)//c)))
axes = np.array(axes).flatten()

for i, col in enumerate(num_cols):
    axes[i].boxplot(df[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor='#1F1F23',color=ORANGE),
                    whiskerprops=dict(color=BLUE,lw=1.2),
                    capprops=dict(color=BLUE,lw=1.2),
                    medianprops=dict(color=GREEN,lw=2),
                    flierprops=dict(marker='o',color=ORANGE,alpha=0.35,markersize=3.5))
    axes[i].set_title(col)
    axes[i].grid(axis='y', alpha=0.4)

for j in range(i+1, len(axes)): axes[j].set_visible(False)
plt.tight_layout()
plt.savefig('graphs and models/outliers_boxplot.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Export Clean Data

In [ ]:
df_clean = df.drop_duplicates()
df_clean.to_csv('eda_cleaned_data.csv', index=False)
df[num_cols].describe().to_csv('eda_stats_summary.csv')

print(f'Raw      : {len(df):,} rows')
print(f'Cleaned  : {len(df_clean):,} rows ({len(df)-len(df_clean)} removed)')
print('Saved    : eda_cleaned_data.csv, eda_stats_summary.csv')

---
Built by Farooq Shah — Data Scientist & ML Engineer — [farooqshah.devsil.com](https://farooqshah.devsil.com)